# Chapter 10 Lab: Reinforcement Learning

```{admonition} Lab Objectives
:class: tip
- Implement Q-learning algorithm
- Build SARSA agent
- Create Deep Q-Network (DQN)
- Apply RL to GridWorld
- Solve CartPole with policy gradients
```

## Lab Overview
1. Tabular Q-learning
2. SARSA (on-policy learning)
3. Function approximation
4. Deep Q-Network
5. Policy gradient methods

## Exercise 1: Q-Learning

Implement tabular Q-learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.state = (0, 0)
        self.goal = (size-1, size-1)
        self.obstacles = {(1, 1), (2, 2), (3, 1)}
        
    def reset(self):
        self.state = (0, 0)
        return self.state
        
    def step(self, action):
        """
        Actions: 0=up, 1=right, 2=down, 3=left
        Return: next_state, reward, done
        """
        x, y = self.state
        moves = [(-1,0), (0,1), (1,0), (0,-1)]
        dx, dy = moves[action]
        new_state = (x+dx, y+dy)
        
        # Check bounds and obstacles
        if (0 <= new_state[0] < self.size and 
            0 <= new_state[1] < self.size and
            new_state not in self.obstacles):
            self.state = new_state
        
        # Reward
        if self.state == self.goal:
            return self.state, 10, True
        else:
            return self.state, -0.1, False

class QLearningAgent:
    def __init__(self, n_actions=4, learning_rate=0.1, discount=0.95, epsilon=0.1):
        self.n_actions = n_actions
        self.alpha = learning_rate
        self.gamma = discount
        self.epsilon = epsilon
        self.q_table = defaultdict(lambda: np.zeros(n_actions))
        
    def get_action(self, state):
        """
        TODO: Implement epsilon-greedy action selection
        With probability epsilon: random action
        Otherwise: argmax Q(s, a)
        """
        # YOUR CODE HERE
        pass
        
    def update(self, state, action, reward, next_state, done):
        """
        TODO: Implement Q-learning update
        
        Q(s,a) ← Q(s,a) + α[r + γ max_a' Q(s',a') - Q(s,a)]
        """
        # YOUR CODE HERE
        pass

# Train
env = GridWorld()
agent = QLearningAgent()

episode_rewards = []
for episode in range(500):
    state = env.reset()
    total_reward = 0
    
    for step in range(100):
        action = agent.get_action(state)
        next_state, reward, done = env.step(action)
        agent.update(state, action, reward, next_state, done)
        
        state = next_state
        total_reward += reward
        
        if done:
            break
    
    episode_rewards.append(total_reward)
    
    if episode % 50 == 0:
        print(f'Episode {episode}, Reward: {total_reward:.2f}')

# Plot learning curve
plt.plot(episode_rewards)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Q-Learning Performance')
plt.show()

# Visualize policy
def visualize_policy(agent, env):
    policy = np.zeros((env.size, env.size))
    for i in range(env.size):
        for j in range(env.size):
            if (i, j) not in env.obstacles:
                policy[i, j] = agent.get_action((i, j))
    
    plt.imshow(policy, cmap='viridis')
    plt.colorbar(label='Action')
    plt.title('Learned Policy')
    plt.show()

visualize_policy(agent, env)

## Exercise 2: SARSA

Implement on-policy SARSA algorithm.

In [ ]:
class SARSAAgent:
    def __init__(self, n_actions=4, learning_rate=0.1, discount=0.95, epsilon=0.1):
        self.n_actions = n_actions
        self.alpha = learning_rate
        self.gamma = discount
        self.epsilon = epsilon
        self.q_table = defaultdict(lambda: np.zeros(n_actions))
        
    def get_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.q_table[state])
        
    def update(self, state, action, reward, next_state, next_action, done):
        """
        TODO: Implement SARSA update
        
        Q(s,a) ← Q(s,a) + α[r + γ Q(s',a') - Q(s,a)]
        
        Key difference from Q-learning: uses actual next action, not max
        """
        # YOUR CODE HERE
        pass

# Train and compare
sarsa_agent = SARSAAgent()
sarsa_rewards = []

for episode in range(500):
    state = env.reset()
    action = sarsa_agent.get_action(state)
    total_reward = 0
    
    for step in range(100):
        next_state, reward, done = env.step(action)
        next_action = sarsa_agent.get_action(next_state)
        sarsa_agent.update(state, action, reward, next_state, next_action, done)
        
        state = next_state
        action = next_action
        total_reward += reward
        
        if done:
            break
    
    sarsa_rewards.append(total_reward)

# Compare Q-learning vs SARSA
plt.figure(figsize=(12, 5))
plt.plot(episode_rewards, label='Q-Learning', alpha=0.7)
plt.plot(sarsa_rewards, label='SARSA', alpha=0.7)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Q-Learning vs SARSA')
plt.legend()
plt.show()

## Exercise 3: Deep Q-Network

Implement DQN with experience replay.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super(DQN, self).__init__()
        """
        TODO: Define network architecture
        Input: state representation
        Output: Q-values for each action
        """
        # YOUR CODE HERE
        pass
        
    def forward(self, x):
        # YOUR CODE HERE
        pass

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
        
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
        
    def sample(self, batch_size):
        """Sample random batch from buffer"""
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return (np.array(state), np.array(action), np.array(reward),
                np.array(next_state), np.array(done))
    
    def __len__(self):
        return len(self.buffer)

class DQNAgent:
    def __init__(self, state_dim, action_dim):
        self.action_dim = action_dim
        self.policy_net = DQN(state_dim, action_dim)
        self.target_net = DQN(state_dim, action_dim)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=0.001)
        self.memory = ReplayBuffer()
        
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.gamma = 0.99
        self.batch_size = 64
        
    def select_action(self, state):
        """Epsilon-greedy action selection"""
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0)
            q_values = self.policy_net(state_t)
            return q_values.argmax().item()
    
    def train_step(self):
        """
        TODO: Implement DQN training step
        
        1. Sample batch from replay buffer
        2. Compute current Q-values
        3. Compute target Q-values (using target network)
        4. Compute loss and update policy network
        """
        if len(self.memory) < self.batch_size:
            return
        
        # YOUR CODE HERE
        pass
    
    def update_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

# Test on CartPole
import gym
env = gym.make('CartPole-v1')
agent = DQNAgent(state_dim=4, action_dim=2)

episode_rewards = []
for episode in range(500):
    state = env.reset()[0]
    total_reward = 0
    
    for t in range(500):
        action = agent.select_action(state)
        next_state, reward, done, truncated, _ = env.step(action)
        done = done or truncated
        
        agent.memory.push(state, action, reward, next_state, done)
        agent.train_step()
        
        state = next_state
        total_reward += reward
        
        if done:
            break
    
    episode_rewards.append(total_reward)
    agent.epsilon = max(agent.epsilon_min, agent.epsilon * agent.epsilon_decay)
    
    if episode % 10 == 0:
        agent.update_target_network()
        print(f'Episode {episode}, Reward: {total_reward:.0f}, Epsilon: {agent.epsilon:.3f}')

plt.plot(episode_rewards)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('DQN on CartPole')
plt.show()

## Exercise 4: Policy Gradient

Implement REINFORCE algorithm.

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        """
        TODO: Define policy network
        Output: probability distribution over actions
        """
        # YOUR CODE HERE
        pass
        
    def forward(self, x):
        # YOUR CODE HERE
        # Use softmax for action probabilities
        pass

class REINFORCEAgent:
    def __init__(self, state_dim, action_dim):
        self.policy = PolicyNetwork(state_dim, action_dim)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=0.01)
        self.gamma = 0.99
        
    def select_action(self, state):
        """Sample action from policy"""
        state_t = torch.FloatTensor(state).unsqueeze(0)
        probs = self.policy(state_t)
        action_dist = torch.distributions.Categorical(probs)
        action = action_dist.sample()
        return action.item(), action_dist.log_prob(action)
    
    def update(self, log_probs, rewards):
        """
        TODO: Implement REINFORCE update
        
        Algorithm:
        1. Compute discounted returns G_t
        2. Policy gradient: ∇ log π(a|s) * G_t
        3. Update policy to maximize expected return
        """
        # YOUR CODE HERE
        pass

# Train
env = gym.make('CartPole-v1')
agent = REINFORCEAgent(state_dim=4, action_dim=2)

episode_rewards = []
for episode in range(1000):
    state = env.reset()[0]
    log_probs = []
    rewards = []
    
    for t in range(500):
        action, log_prob = agent.select_action(state)
        next_state, reward, done, truncated, _ = env.step(action)
        done = done or truncated
        
        log_probs.append(log_prob)
        rewards.append(reward)
        state = next_state
        
        if done:
            break
    
    agent.update(log_probs, rewards)
    episode_rewards.append(sum(rewards))
    
    if episode % 50 == 0:
        print(f'Episode {episode}, Reward: {sum(rewards):.0f}')

plt.plot(episode_rewards)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('REINFORCE on CartPole')
plt.show()

## Exercise 5: Compare RL Algorithms

Benchmark all algorithms.

In [ ]:
def train_and_evaluate(agent_class, env_name, n_episodes=500):
    """
    TODO: Standard training and evaluation protocol
    Return: episode rewards, final performance
    """
    # YOUR CODE HERE
    pass

# Compare all algorithms
results = {}
algorithms = {
    'Q-Learning': QLearningAgent,
    'SARSA': SARSAAgent,
    'DQN': DQNAgent,
    'REINFORCE': REINFORCEAgent
}

for name, agent_class in algorithms.items():
    print(f'Training {name}...')
    rewards = train_and_evaluate(agent_class, 'CartPole-v1')
    results[name] = rewards

# Visualize comparison
plt.figure(figsize=(12, 6))
for name, rewards in results.items():
    # Smooth with moving average
    smoothed = np.convolve(rewards, np.ones(20)/20, mode='valid')
    plt.plot(smoothed, label=name, alpha=0.7)

plt.xlabel('Episode')
plt.ylabel('Reward (smoothed)')
plt.title('RL Algorithm Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Performance summary
for name, rewards in results.items():
    final_100 = np.mean(rewards[-100:])
    print(f'{name:15s}: Final 100 episodes average = {final_100:.1f}')

## Challenge: Actor-Critic

Implement Advantage Actor-Critic (A2C).

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()
        """
        CHALLENGE: Implement Actor-Critic
        
        Actor: outputs action probabilities
        Critic: outputs value function V(s)
        
        Advantage: A(s,a) = r + γV(s') - V(s)
        """
        # YOUR CODE HERE
        pass
        
    def forward(self, x):
        # Return both action probs and value
        # YOUR CODE HERE
        pass

## Lab Report

### Deliverables
- [ ] Q-learning implementation
- [ ] DQN with experience replay
- [ ] Policy gradient (REINFORCE)
- [ ] Algorithm comparison
- [ ] Analysis of convergence

### Discussion
1. Compare on-policy vs off-policy learning
2. Why is experience replay important for DQN?
3. When do policy gradients work better than Q-learning?
4. How do you balance exploration and exploitation?